## Tokenization
Sebeum model AI bisa memproses kalimat, mereka harus memecah teks tersebut menjadi bagian-bagian kecil yang disebut Tokens. Token bisa berupa kata utuh, potongan kata, atau bahkan tanda baca. Disini akan dgunakan library tiktoken untuk melihat bagaimana sebuah kalimat diubah menjadi angka-angka (Token IDs). 

In [14]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi namaku qi, aku suka bermain game")

In [15]:
tokens

[12194, 5165, 9385, 64017, 11, 26126, 37041, 53711, 2813]

Setiap angka di atas mewakili satu potongan kata atau karakter. Mari kita pecah dan kembalikan angka-angka tersebut menjadi teks asli (*decoding*) untuk melihat dengan jelas bagaimana AI memotong kalimat tersebut.

In [16]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
5165 =  nam
9385 = aku
64017 =  qi
11 = ,
26126 =  aku
37041 =  suka
53711 =  bermain
2813 =  game


In [18]:
encoding.decode([11])

','

Pernahkah kita merasa bahwa AI ingat apa yang kita katakan di awal percakapan? Ternyata, itu hanyalah sebuah ilusi!

Mari kita siapkan *environment* dan API key terlebih dahulu untuk membuktikannya.

In [19]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("Tidak ditemukan API key")
elif not api_key.startswith("sk-proj-"):
    print("API key tidak valid")
else:
    print("API key valid")

API key valid


In [20]:
from openai import OpenAI

openai = OpenAI()

Disini kita akan memulai percakapan dengan AI. Di pesan pertama ini, kita akan memperkenalkan diri kita.

In [21]:
messages = [
    {"role": "system", "content": "Kamu adalah asisten yang membantu menjawab pertanyaan."},
    {"role": "user", "content": "Hai!, Aku qi."}
    ]

In [22]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Hai qi! Ada yang bisa aku bantu hari ini?'

Kita baru saja memberitahunya! Apa yang terjadi??

Inilah rahasianya: setiap panggilan ke LLM bersifat sepenuhnya **STATELESS** (tidak menyimpan data atau memori sebelumnya). Setiap panggilan adalah interaksi yang benar-benar baru. Sebagai *AI engineers*, adalah TUGAS KITA untuk merancang teknik yang memberikan kesan bahwa LLM memiliki "ingatan". 

Caranya adalah dengan menyisipkan seluruh riwayat percakapan ke dalam pemanggilan API yang baru.

In [29]:
messages = [
    {"role": "system", "content": "Kamu adalah asisten yang membantu menjawab pertanyaan."},
    {"role": "user", "content": "Hai! Aku qi."},
    {"role": "assistant", "content": "Hai qi! Ada yang bisa aku bantu hari ini?"},
    {"role": "user", "content": "Siapa namaku?"}
    ]

In [30]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Namamu adalah qi, kan? Ada yang ingin kamu tanyakan atau butuh bantuan apa?'

Sekarang, mari kita buat panggilan baru ke AI dan langsung menanyakan siapa nama kita, **tanpa** memberikan riwayat obrolan sebelumnya. Mari kita lihat apakah dia ingat!

In [27]:
messages_lupa = [
    {"role": "system", "content": "Kamu adalah asisten yang membantu menjawab pertanyaan."},
    {"role": "user", "content": "Siapa namaku?"}
    ]

In [28]:
response_lupa = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages_lupa)
response_lupa.choices[0].message.content

'Maaf, saya tidak tahu siapa namamu karena kamu belum memberitahukannya. Bisa tolong beri tahu namamu?'

## Catatan singkat
### The Catch: Context Windows and Token Limits

Jika kita harus mengirim seluruh riwayat percakapan setiap saat agar AI "ingat", apakah ada batasannya? Jawabannya: **YA**.

Setiap model AI memiliki batasan kapasitas yang disebut **Context Window** atau batas kata (*word limit*). Misalnya, sebuah model mungkin hanya bisa memproses maksimal 8.000, 16.000, atau 128.000 token dalam satu kali panggilan API (*API call*). 

Jika percakapan kita sangat panjang hingga total tokennya melebihi batas *Context Window* ini, sistem harus mulai memotong atau "melupakan" pesan-pesan yang paling awal (paling lama). Konteks obrolan lama akan tergeser agar pesan terbaru bisa muat diproses oleh model.

### Does the AI only read the last line?

Sebuah pertanyaan logis sering muncul: Saat kita mengirim ulang seluruh riwayat percakapan yang panjang itu, apakah AI hanya melihat baris terakhirnya saja?

**Jawabannya: Tidak.** Setiap kali riwayat percakapan dikirim, AI benar-benar memproses ulang **seluruh teks dari awal sampai akhir**. 

**Analogi Amnesia:**
Bayangkan AI sebagai seseorang yang memiliki amnesia jangka pendek, tetapi ditugaskan untuk menulis lanjutan dari sebuah cerita bersambung. Setiap kali tiba gilirannya untuk menulis, ia harus membaca tumpukan kertas cerita tersebut dari halaman pertama hingga kalimat terakhir. Setelah ia memahami seluruh konteks ceritanya kembali, barulah ia bisa memprediksi dan menulis kata selanjutnya. 

Inilah mengapa pemrosesan teks AI (LLM) membutuhkan daya komputasi yang besar. Semakin panjang percakapan yang dikirim ulang, semakin banyak token yang harus dibaca, dan semakin besar pula biaya komputasinya!